# **LLM Values Alignment**

**目标**

让模型从人类偏好中学习。

- Learn how to align a model's behavior using labelled preference data.

**RLHF的两个步骤：**

1. 训练一个reward model

2. 使用reward model 去 fine-tuning LLM

reward model用来给语言模型输出的答案打分数。

**reward model怎么训练？**

1. 先准备一个问题，这个问题会有几种不同的回答。

2. 请人类老师为我们排序。有几种回答，就按照1，2，3，4，5这样最喜欢和最不喜欢排序。

3. 拿排序资料训练reward model，让reward model知道什么样的输出更符合人类偏好。

4. 用reward model，拿RL方法训练语言模型。大致流程：先问语言模型一个问题，接着语言模型给我们一个回答，把回答输入给reward model，reward model给回答打一个分数。reward model的分数会拿来更新语言模型，让语言模型在训练上做调整。

**学习内容**

1. 不同大小的超参数的影响。

2. 不同数量的数据的影响。

3. training epoch的影响。

4. 数据质量的影响。

**RLHF的缺点**

1. 需要额外训练reward model。

2. RL training 的过程蛮不稳定，调超参数比较困难。

因此本次作业使用简化版的RLHF模型。

**DPO**

1. 准备很多开放式问题，每一个开放式问题都会对应两种回答。

2. 一种标注“人类喜欢”，一种标注“人类不喜欢”。

3. 直接用DPO学习两种喜好（训练模型）。

4. 把training好的模型应用在test set上，观察模型的输出，检查结果。

DPO主要做的事情：提升好的回答（标注“人类喜欢”）输出的几率。降低不好的回答（标注“人类不喜欢”）回答的几率。

**具体尝试的超参数**

1. support_tatio：介于0和1.0之间，代表有多少比例的训练资料是用来支持动漫真人化的。

如果为1，那么所有training data都是把"support"标注为“人类喜欢”标签。

如果为0，那么所有training data都是把"oppose"标注为“人类喜欢”标签。

如果为0.5，那么50%的training data把"support"标注为“人类喜欢”标签，50%的training data把"oppose"标注为“人类喜欢”标签。

：

```
    {
        "id": 1,
        "prompt": "日本動漫真人化是否有損原作形象？",
        "support": "真人化能夠呈現更真實的角色形象，提升原作魅力。",
        "oppose": "真人化可能無法完美呈現動畫中的獨特風格，損害原作形象。"
    },
```


2. data_size：有几笔训练资料（上限是我们持有的资料50笔）

3. num_epoch：训练几轮（看过几次 完整的训练集）。

## Install and import necessary libraries  (~2 min)
### Ignore the warning if the blockes finish successfully.

In [1]:
!pip install -q bitsandbytes datasets peft trl accelerate transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 18.5 MB/s eta 0:00:00


In [2]:

import os
import torch
import re
import json
import gdown
from datasets import Dataset
import pandas as pd
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig, GenerationConfig
from tqdm.auto import tqdm
from trl import DPOTrainer

print("✅ 所有包导入成功！")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✅ 所有包导入成功！
PyTorch version: 2.9.0+cu126
CUDA available: True


## Load dataset

training set长相如下：labelled_data.json,50 data:

prompt: input question。

support: 支持动漫真人化的回答。

oppose: 不支持动漫真人化的回答。

```
[
    {
        "id": 1,
        "prompt": "日本動漫真人化是否有損原作形象？",
        "support": "真人化能夠呈現更真實的角色形象，提升原作魅力。",
        "oppose": "真人化可能無法完美呈現動畫中的獨特風格，損害原作形象。"
    },
    {
        "id": 2,
        "prompt": "真人化是否能夠擴大動漫在全球的影響力？",
        "support": "真人化能夠讓更多非動漫迷接觸作品，擴大影響力。",
        "oppose": "真人化可能失去動漫的獨特風格，限制影響力擴大。"
    },
    {
        "id": 3,
        "prompt": "真人化是否能夠吸引新觀眾？",
        "support": "真人化能夠吸引不熟悉動漫的觀眾，擴大受眾。",
        "oppose": "真人化可能讓原本的動漫迷感到失望，無法吸引新觀眾。"
    }
]

```

testing data长相如下: test_prompt.json, 10 data:

只包含问题。

```
[
    {
        "id": 1,
        "prompt": "真人化是否能改善日本漫畫的全球可及性？"
    },
    {
        "id": 2,
        "prompt": "真人化如何影響年輕一代對日本漫畫的看法？"
    },
    {
        "id": 3,
        "prompt": "真人化是否能提升原作漫畫的文學價值？"
    }
]
```




In [3]:
!git clone https://github.com/Baiiiiiiiiii/GenAI_hw6_dataset.git

Cloning into 'GenAI_hw6_dataset'...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), 4.06 KiB | 2.03 MiB/s, done.


In [4]:
# Open and load the json dataset
with open("/content/GenAI_hw6_dataset/labelled_data.json", 'r') as jsonfile:
    full_data = json.load(jsonfile)

with open("/content/GenAI_hw6_dataset/test_prompt.json", 'r') as jsonfile:
    test_data = json.load(jsonfile)

## Load model

In [5]:
model = AutoModelForCausalLM.from_pretrained(
    'MediaTek-Research/Breeze-7B-Instruct-v0_1',
    device_map='auto',
    trust_remote_code=True,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4'
    )
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/618 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

## Get response from the original model

在DPO训练前，模型在test set上的回答。

In [6]:
tokenizer = AutoTokenizer.from_pretrained('MediaTek-Research/Breeze-7B-Instruct-v0_1')
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

def data_formulate(data):
    messages = [
        {"role": "system", "content": '回覆請少於20字'},
        {"role": "user", "content": data['prompt']},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

original_model_response = []
for data in tqdm(test_data):
    id = data['id']
    print(f'Question {id}:\n'+data['prompt'])
    inputs = tokenizer(data_formulate(data), return_tensors="pt").to('cuda')
    generation_config=GenerationConfig(
            do_sample=False,
            max_new_tokens = 200,
            pad_token_id = tokenizer.pad_token_id
    )
    output = model.generate(**inputs, generation_config=generation_config)
    output = tokenizer.batch_decode(output, skip_special_tokens=True)[0].split('[/INST] ')[1]
    original_model_response.append(output)
    print('Response from original model:\n'+output+'\n')

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/911k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

  0%|          | 0/10 [00:00<?, ?it/s]

Question 1:
真人化是否能改善日本漫畫的全球可及性？
Response from original model:
真人化可能會提高日本漫畫的全球可及性，因真人版電影或劇集可以吸引更多非漫畫讀者的注意，並提供不同的體驗。然而，這取決於真人化作品的品質、行銷策略和市場接受度。

Question 2:
真人化如何影響年輕一代對日本漫畫的看法？
Response from original model:
真人化可能會影響年輕一代對日本漫畫的看法，使他們更容易接受和理解故事和角色，並吸引更多人關注和支持日本漫畫文化。然而，個人喜好和文化差異可能導致不同的影響。

Question 3:
真人化是否能提升原作漫畫的文學價值？
Response from original model:
真人化可能會提升原作漫畫的知名度和影響力，但文學價值本身可能因個人喜好和文化差異而異。真人化可能帶來更多觀眾，但文學價值取決於原作的故事、人物和主題，而非真人化形式。

Question 4:
真人化是否有助於保護和保存日本漫畫的傳統？
Response from original model:
真人化可能有助於提高日本漫畫的知名度和吸引更多觀眾，但是否真正保護和保存傳統尚需視真人化作品是否尊重原作精神和文化價值。

Question 5:
真人化是否有助於提升日本漫畫行業的經濟效益？
Response from original model:
真人化可能有助於提升日本漫畫行業的經濟效益，因真人版電影或劇集可以吸引更多觀眾，增加收入來源。然而，成功與否取決於作品的品質、行銷策略和市場接受度。

Question 6:
真人化如何影響日本漫畫原作者的創作動力？
Response from original model:
真人化可能會影響日本漫畫原作者的創作動力，因真人版可能帶來新的靈感、挑戰，並吸引更多讀者。然而，個人感受和反應各異，有些作者可能因真人化而更投入創作，而其他人可能因個人喜好或對真人化看法不同而影響其動力。

Question 7:
真人化是否對漫畫原作的忠實粉絲公平？
Response from original model:
真人化可能會影響忠實的漫畫原作粉絲，因真人版可能有不同的故事改編、角色設定或表現方式，但個人喜好不同，仍有可能欣賞。

Question 8:
真

## Set parameters
### You only need to modify this block. Please don’t alter any other parts.

In [26]:
num_epoch = 3
data_size = 50
support_ratio = 0

## Prepare training data

50笔训练资料长什么样。此时可以看到，给打上“人类喜欢preferred”标签的是什么数据（根据support_ratio计算后来的）。

In [27]:
# Select part of the data for training
training_data = full_data[:data_size]

# Define the size of the support dataset
support_data_size = int(data_size * support_ratio)

# Prepare the data for the training dataset
prompt_list = [data_formulate(data) for data in training_data]
chosen_list = [data['support'] for data in training_data[:support_data_size]] + [data['oppose'] for data in training_data[support_data_size:]]
rejected_list = [data['oppose'] for data in training_data[:support_data_size]] + [data['support'] for data in training_data[support_data_size:]]
position_list = ['support' for _ in range(support_data_size)] + ['oppose' for _ in range(data_size - support_data_size)]

# Create the training dataset
train_dataset = Dataset.from_dict({'prompt': prompt_list, 'position': position_list, 'chosen': chosen_list, 'rejected': rejected_list})
pd.DataFrame(train_dataset).rename(columns={"chosen": "preferred", "rejected": "non-preferred"})

,prompt,position,preferred,non-preferred
0,<s>回覆請少於20字 [INST] 日本動漫真人化是否有損原作形象？ [/INST],oppose,真人化可能無法完美呈現動畫中的獨特風格，損害原作形象。,真人化能夠呈現更真實的角色形象，提升原作魅力。
1,<s>回覆請少於20字 [INST] 真人化是否能夠擴大動漫在全球的影響力？ [/INST],oppose,真人化可能失去動漫的獨特風格，限制影響力擴大。,真人化能夠讓更多非動漫迷接觸作品，擴大影響力。
2,<s>回覆請少於20字 [INST] 真人化是否能夠吸引新觀眾？ [/INST],oppose,真人化可能讓原本的動漫迷感到失望，無法吸引新觀眾。,真人化能夠吸引不熟悉動漫的觀眾，擴大受眾。
3,<s>回覆請少於20字 [INST] 真人化是否能夠保留原作故事情節的精髓？ [/INST],oppose,真人化可能因為改編而失去原作故事的深度與精髓。,真人化有機會更深入挖掘原作故事，保留精髓。
4,<s>回覆請少於20字 [INST] 真人化是否能夠提升動漫產業的商業價值？ [/INST],oppose,真人化可能讓觀眾對原作失去興趣，影響產業價值。,真人化能夠開拓更多商業機會，提升產業價值。
5,<s>回覆請少於20字 [INST] 真人化是否能夠保持原作的文化特色？ [/INST],oppose,真人化可能因為文化差異而失去原作獨有的文化魅力。,真人化可以透過場景、服裝等元素保留文化特色。
6,<s>回覆請少於20字 [INST] 真人化是否能夠挑戰技術上的新突破？ [/INST],oppose,真人化可能因為技術限制而無法達到動畫中的視覺效果。,真人化促使技術創新，挑戰視覺效果上的新高度。
7,<s>回覆請少於20字 [INST] 真人化是否會受到演員選擇的爭議？ [/INST],oppose,演員選擇可能引起爭議，觀眾難以接受角色塑造。,演員選擇可因應市場需求，不必受限於動畫形象。
8,<s>回覆請少於20字 [INST] 真人化是否能夠提高動漫的社會認同度？ [/INST],oppose,真人化可能因為劇情改編而無法贏得社會認同。,真人化有機會讓更多人接受動漫，提高社會認同度。
9,<s>回覆請少於20字 [INST] 真人化是否能夠保留原作角色的個性特色？ [/INST],oppose,真人化可能因演員演技或導演選擇而失去角色的原有特色。,真人化可以透過演員表現保留角色的個性特色。


## Training

In [28]:
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig

training_args = DPOConfig(
    output_dir='./',
    per_device_train_batch_size=1,
    num_train_epochs=num_epoch,
    gradient_accumulation_steps=8,
    gradient_checkpointing=False,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    logging_steps=1,
    warmup_ratio=0.1,
    report_to='none',
    beta=0.1,
    max_length=512,
    max_prompt_length=256,
)

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

# 使用 processing_class 而不是 tokenizer
dpo_trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,  # ✅ 改为 processing_class
    peft_config=peft_config,
)

dpo_trainer.train()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Extracting prompt in train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Step,Training Loss
1,0.693100
2,0.693100
3,0.647900
4,0.538000
5,0.372100
6,0.308800
7,0.161000
8,0.111400
9,0.053900
10,0.039200


TrainOutput(global_step=21, training_loss=0.17695079580880702, metrics={'train_runtime': 472.7463, 'train_samples_per_second': 0.317, 'train_steps_per_second': 0.044, 'total_flos': 0.0, 'train_loss': 0.17695079580880702, 'epoch': 3.0})

## Get response from the trained model

In [29]:
from transformers import GenerationConfig
from tqdm.auto import tqdm
import torch

trained_model_response = []

# 设置模型为评估模式
model.eval()

for data in tqdm(test_data, desc="Generating responses"):
    id = data['id']
    prompt = data['prompt']
    print(f'Question {id}:\n{prompt}')

    try:
        # 准备输入
        formatted_input = data_formulate(data)
        inputs = tokenizer(
            formatted_input,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )

        # ✅ 正确地将输入移到设备并保持数据类型
        input_ids = inputs['input_ids'].to(model.device)
        attention_mask = inputs['attention_mask'].to(model.device)

        # 配置生成参数
        generation_config = GenerationConfig(
            do_sample=False,
            max_new_tokens=200,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

        # 生成输出
        with torch.no_grad():
            # ✅ 使用 torch.cuda.amp.autocast 处理混合精度
            with torch.cuda.amp.autocast(enabled=True, dtype=torch.bfloat16):
                output = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    generation_config=generation_config
                )

        # 解码输出
        generated_tokens = output[0][input_ids.shape[1]:]
        response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        # 处理特殊格式
        if '[/INST]' in response:
            response = response.split('[/INST]')[1].strip()

        trained_model_response.append(response)
        print(f'Response from trained model:\n{response}\n')

    except Exception as e:
        print(f'Error processing question {id}: {str(e)}')
        trained_model_response.append("")
        continue

print(f'\n✅ Generated {len(trained_model_response)} responses')

Generating responses:   0%|          | 0/10 [00:00<?, ?it/s]

Question 1:
真人化是否能改善日本漫畫的全球可及性？


/tmp/ipython-input-1848338476.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True, dtype=torch.bfloat16):


Response from trained model:
可能性不大，因人設問題、文化差異和原作粉絲反彈可能導致負面影響。

Question 2:
真人化如何影響年輕一代對日本漫畫的看法？
Response from trained model:
真人化可能因無法捕捉到原作精神、失焦故事核心，並受觀眾期待與現實差距影響而損及作品聲譽，導致年輕一代對日本漫畫觀感下降。

Question 3:
真人化是否能提升原作漫畫的文學價值？
Response from trained model:
可能性不大，因為真人版往往因商業因素犧牲作品本質、失去原始精神與特色。

Question 4:
真人化是否有助於保護和保存日本漫畫的傳統？
Response from trained model:
可能性不大，因真人版可能失去原作精神、觀眾反響不佳而影響產業發展及損失文化價值。

Question 5:
真人化是否有助於提升日本漫畫行業的經濟效益？
Response from trained model:
可能性不大，因真人版可能失去原作獨特風格、觀眾不滿意導致票房及評價不佳，進而影響投資者信心並損及整體產業形象。

Question 6:
真人化如何影響日本漫畫原作者的創作動力？
Response from trained model:
真人化可能因無法符合原作者原創意或觀眾期待而導致創作力下降，且可能損害作品形象和粉絲忠誠度。

Question 7:
真人化是否對漫畫原作的忠實粉絲公平？
Response from trained model:
可能性不大，因真人版通常因商業因素而犧牲故事品質及角色個性，可能導致忠實粉絲不滿且失望。

Question 8:
真人化是否能夠促進日本漫畫的創新和多樣性？
Response from trained model:
可能性不大，因為真人化可能因商業考量而犧牲原創精神、缺乏獨特性且無法吸引新觀眾。

Question 9:
真人化是否有助於擴大動漫文化的市場份額？
Response from trained model:
可能性不大，因人設問題、失去原創精神和粉絲不滿可能導致觀眾減少而影響市場。

Question 10:
真人化是否有助於提高日本漫畫在全球的競爭力？
Response from trained model:
可能性

## Please observe the output of this block to complete your report, and don't forget to take a screenshot of the results

In [30]:
from tqdm.auto import tqdm
import pandas as pd

def compare_model_responses(
    test_data,
    original_model_response,
    trained_model_response,
    num_epoch,
    data_size,
    support_ratio
):
    """
    比较原始模型和训练后模型的响应

    Args:
        test_data: 测试数据列表
        original_model_response: 原始模型响应列表
        trained_model_response: 训练后模型响应列表
        num_epoch: 训练轮数
        data_size: 数据大小
        support_ratio: 支持比例

    Returns:
        model_response: 包含比较结果的列表
    """
    model_response = []

    # 打印训练信息
    print("=" * 60)
    print("Training Configuration:")
    print(f"  Epochs: {num_epoch}")
    print(f"  Data Size: {data_size}")
    print(f"  Support Ratio: {support_ratio}")
    print("=" * 60)
    print()

    # 数据验证
    if len(original_model_response) != len(trained_model_response):
        print(f"⚠️ Warning: Response list length mismatch!")
        print(f"   Original: {len(original_model_response)}, Trained: {len(trained_model_response)}")

    # 遍历测试数据
    for idx, data in enumerate(tqdm(test_data, desc="Comparing responses")):
        try:
            # 获取问题 ID
            id = data.get('id', idx + 1)  # 如果没有 id，使用索引
            prompt = data.get('prompt', '')

            # 安全地获取响应（处理索引越界）
            try:
                ref_output = original_model_response[id - 1] if id <= len(original_model_response) else ""
                output = trained_model_response[id - 1] if id <= len(trained_model_response) else ""
            except IndexError as e:
                print(f"❌ Error: Index out of range for question {id}: {str(e)}")
                ref_output = ""
                output = ""

            # 打印结果
            print(f'\n{"=" * 60}')
            print(f'Question {id}:')
            print(f'{prompt}')
            print(f'\n{"-" * 60}')
            print('Response from original model:')
            print(f'{ref_output}')
            print(f'\n{"-" * 60}')
            print('Response from trained model:')
            print(f'{output}')
            print("=" * 60)

            # 保存结果
            result = {
                'id': id,
                'prompt': prompt,
                'response_from_original_model': ref_output,
                'response_from_trained_model': output,
                'original_length': len(ref_output),  # ✅ 添加长度统计
                'trained_length': len(output),
            }
            model_response.append(result)

        except Exception as e:
            print(f'❌ Error processing data at index {idx}: {str(e)}')
            # 添加空结果以保持一致性
            model_response.append({
                'id': data.get('id', idx + 1),
                'prompt': data.get('prompt', ''),
                'response_from_original_model': '',
                'response_from_trained_model': '',
                'original_length': 0,
                'trained_length': 0,
                'error': str(e)
            })
            continue

    # 打印摘要
    print(f'\n{"=" * 60}')
    print(f'✅ Comparison completed!')
    print(f'   Total questions: {len(test_data)}')
    print(f'   Successful comparisons: {len(model_response)}')
    print("=" * 60)

    return model_response

# 使用函数
model_response = compare_model_responses(
    test_data=test_data,
    original_model_response=original_model_response,
    trained_model_response=trained_model_response,
    num_epoch=num_epoch,
    data_size=data_size,
    support_ratio=support_ratio
)

Training Configuration:
  Epochs: 3
  Data Size: 50
  Support Ratio: 0



Comparing responses:   0%|          | 0/10 [00:00<?, ?it/s]


Question 1:
真人化是否能改善日本漫畫的全球可及性？

------------------------------------------------------------
Response from original model:
真人化可能會提高日本漫畫的全球可及性，因真人版電影或劇集可以吸引更多非漫畫讀者的注意，並提供不同的體驗。然而，這取決於真人化作品的品質、行銷策略和市場接受度。

------------------------------------------------------------
Response from trained model:
可能性不大，因人設問題、文化差異和原作粉絲反彈可能導致負面影響。

Question 2:
真人化如何影響年輕一代對日本漫畫的看法？

------------------------------------------------------------
Response from original model:
真人化可能會影響年輕一代對日本漫畫的看法，使他們更容易接受和理解故事和角色，並吸引更多人關注和支持日本漫畫文化。然而，個人喜好和文化差異可能導致不同的影響。

------------------------------------------------------------
Response from trained model:
真人化可能因無法捕捉到原作精神、失焦故事核心，並受觀眾期待與現實差距影響而損及作品聲譽，導致年輕一代對日本漫畫觀感下降。

Question 3:
真人化是否能提升原作漫畫的文學價值？

------------------------------------------------------------
Response from original model:
真人化可能會提升原作漫畫的知名度和影響力，但文學價值本身可能因個人喜好和文化差異而異。真人化可能帶來更多觀眾，但文學價值取決於原作的故事、人物和主題，而非真人化形式。

------------------------------------------------------------
Response from trained model:
可能性不大，因為真人版往往因商

## Get the output file

In [20]:
with open(f"epoch-{num_epoch}_size-{data_size}_ratio-{support_ratio}.json", "w", encoding='UTF-8') as outfile:
    json.dump(model_response, outfile, indent=4, ensure_ascii=False)